# Leads CSV — filter by Google `types`

Loads a scraper results CSV (same columns as `krakow_leads.csv`). The `types` column is pipe-separated (`coffee_shop|bar|cafe`).

**Exclusions** match **whole tokens** between `|` — so dropping `bar` removes rows that include the `bar` place type, but **not** `barber_shop`.

Requirements: `pandas` (already in this repo’s `requirements.txt`). For interactive widgets in the last section, run `pip install ipywidgets` once.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Default matches API scraper output; change if needed
CSV_PATH = Path("krakow_leads.csv")

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows from {CSV_PATH.resolve()}")
df.head()

Loaded 1,336 rows from /Users/bartek/Documents/GitHub/places-scraper/krakow_leads.csv


,name,area,address,phone,rating,reviews,types,website,score,google_maps_link,place_id
0,Gossip Cafe,Stare Miasto – płn.-zach.; Stare Miasto – płn....,"Świętego Jana 30, 31-018 Kraków",NaN,4.7,3647.0,breakfast_restaurant|cafe|bakery|food_store|re...,http://www.facebook.com/gossipcafekrakow,13,https://maps.google.com/?cid=10370715884191885...,ChIJH5wflXJbFkcRHyfCdyYv7I8
1,Tociekawa Specialty Coffee,Stare Miasto – płn.-zach.; Stare Miasto – płn....,"Długa 37, 31-147 Kraków",NaN,4.8,2479.0,cafe|lounge_bar|coffee_shop|bakery|night_club|...,https://facebook.com/tociekawa/,13,https://maps.google.com/?cid=11419981840062961...,ChIJ7_9XZsBbFkcR9lddVvrse54
2,Dolce far niente,Stare Miasto – płn.-zach.; Stare Miasto – płn....,"Czysta 8/LU1, 31-121 Kraków",728 618 015,4.8,852.0,coffee_shop|wine_bar|cafe|bakery|food_store|ba...,NaN,13,https://maps.google.com/?cid=49320309469530644...,ChIJ-SAEjRdbFkcRE_Be5gMYckQ
3,KRUSZ,Stare Miasto – płn.-zach.; Stare Miasto – płn....,"Ignacego Paderewskiego 4, 31-157 Kraków",570 004 586,4.5,648.0,coffee_shop|pastry_shop|cafe|dessert_shop|bake...,https://instagram.com/krusz.krakow?igshid=MmIz...,13,https://maps.google.com/?cid=76846732225603053...,ChIJ-f96dI5bFkcR4dCprZRxpWo
4,Fable Cafe,Stare Miasto – płn.-zach.; Stare Miasto – płn....,"Rajska 20, 31-124 Kraków",536 443 378,4.7,1260.0,coffee_shop|brunch_restaurant|breakfast_restau...,https://www.instagram.com/fable_cafe,13,https://maps.google.com/?cid=11985144895253151...,ChIJLbw-HqBbFkcRW30mP8jJU6Y


## Discover type tokens
Use this to pick tokens to exclude (sorted by frequency).

In [2]:
type_tokens = (
    df["types"].dropna().astype(str).str.split("|").explode().str.strip()
)
type_counts = type_tokens.value_counts()
display(type_counts.head(50))
print(f"… {len(type_counts):,} distinct tokens total")

types
point_of_interest           1336
establishment               1336
store                        682
service                      426
food                         347
food_store                   240
health                       228
bar                          147
restaurant                   142
florist                      137
hair_care                    132
bakery                       129
hair_salon                   116
cafe                         114
confectionery                114
beauty_salon                 110
dessert_shop                  93
body_art_service              92
coffee_shop                   77
pharmacy                      69
sports_activity_location      61
hotel                         61
lodging                       61
gym                           59
night_club                    59
nail_salon                    52
medical_clinic                51
pub                           49
ice_cream_shop                48
jewelry_store                 48
car_

… 174 distinct tokens total


## Exclude rows by type tokens
Edit `EXCLUDED_TYPES`. Rows where **any** listed token appears in `types` are removed from `filtered`.

In [19]:
EXCLUDED_TYPES = {
    "bar",
    "food"
    # "pub",
    # "night_club",
    # "wine_bar",
}


def row_has_excluded_type(types_val, excluded: set[str]) -> bool:
    if pd.isna(types_val):
        return False
    s = str(types_val).strip()
    if not s:
        return False
    tokens = {t.strip() for t in s.split("|") if t.strip()}
    return bool(tokens & excluded)


mask_drop = df["types"].apply(lambda v: row_has_excluded_type(v, EXCLUDED_TYPES))
filtered = df.loc[~mask_drop].copy()
print(
    f"Excluded {mask_drop.sum():,} rows matching {sorted(EXCLUDED_TYPES)!r}; "
    f"{len(filtered):,} rows left"
)
filtered

Excluded 417 rows matching ['bar', 'food']; 919 rows left


,name,area,address,phone,rating,reviews,types,website,score,google_maps_link,place_id
8,Empire of Beauty,Stare Miasto – płn.-zach.; Stare Miasto – płn....,"plac Szczepański 3, 31-011 Kraków",532 281 363,4.8,1284.0,beauty_salon|hair_salon|nail_salon|hair_care|p...,https://instagram.com/empire_of_beauty_krakow?...,13,https://maps.google.com/?cid=73481987926912340...,ChIJtRmh6z1bFkcR7rRQmd4L-mU
9,TOP NAILS SPA,Stare Miasto – płn.-zach.; Stare Miasto – płn....,"Rynek Główny 6, 30-041 Kraków",691 069 370,4.7,414.0,nail_salon|spa|skin_care_clinic|medical_clinic...,NaN,13,https://maps.google.com/?cid=14222804339331509...,ChIJ_9n--BFbFkcRAiy94zCNYcU
10,Kościuszki 17,Stare Miasto – płn.-zach.; Kazimierz – środek;...,"Gen, Tadeusza Kościuszki, 30-105 Kraków",664 880 430,4.5,309.0,nail_salon|beauty_salon|point_of_interest|serv...,https://m.facebook.com/kosciuszki17krakow,13,https://maps.google.com/?cid=27245233722489355...,ChIJFX9CLQhbFkcRlqwlaZJzzyU
11,Beauty Hall,Stare Miasto – płn.-zach.; Stare Miasto – płn....,"Wielopole 16, 31-072 Kraków",735 950 700,4.6,255.0,nail_salon|beauty_salon|point_of_interest|serv...,https://www.instagram.com/beauty_hall_krakow,13,https://maps.google.com/?cid=54262841576147966...,ChIJL731MshbFkcRaa_1ELEITks
12,Królewska Studio,Stare Miasto – płn.-zach.; Krowodrza – Błonia;...,"Urzędnicza 47/3, 30-048 Kraków",887 199 100,4.7,203.0,barber_shop|skin_care_clinic|nail_salon|medica...,https://www.facebook.com/Kr%C3%B3lewska-Studio...,13,https://maps.google.com/?cid=75031246206909847...,ChIJWfsni6VbFkcRMA_G2RZ0IGg
...,...,...,...,...,...,...,...,...,...,...,...
1331,Park street workout,Prądnik Biały,"Białoprądnicka, 31-221 Kraków",NaN,3.0,4.0,gym|sports_activity_location|health|point_of_i...,NaN,5,https://maps.google.com/?cid=34976918377372571...,ChIJadL5JIVbFkcRxsgc4jFMijA
1332,W80 Apartment,Prądnik Biały,"Wrocławska 80, 31-000 Kraków",NaN,NaN,NaN,hotel|lodging|point_of_interest|establishment,NaN,5,https://maps.google.com/?cid=16242767484617045...,ChIJu5frRr5bFkcRLDXdKfjmaeE
1333,Industry Suppliers,Prądnik Biały,"Opolska 14A, 31-323 Kraków",12 376 37 00,NaN,NaN,hotel|lodging|point_of_interest|establishment,NaN,5,https://maps.google.com/?cid=11943080155104066...,ChIJx7-zQkhaFkcR8rr-uB9YvqU
1334,"Centrum osteopatyczne, biodynamiczne, fizjoter...",Prądnik Biały,"Pleszowska 23, 31-228 Kraków",690 230 003,NaN,NaN,hotel|spa|lodging|point_of_interest|establishment,NaN,5,https://maps.google.com/?cid=79191264052604526...,ChIJgSLreYZbFkcRVFCyz1lw_Qo


### Optional: substring exclusions
If you really want “contains the substring `bar` anywhere” (this will also match `barber_shop`, `wine_bar`, etc.), set non-empty `EXCLUDED_SUBSTRINGS` and run the next cell instead of the token-based cell above.

In [ ]:
EXCLUDED_SUBSTRINGS: tuple[str, ...] = ()  # e.g. ("bar",) — use with care

if EXCLUDED_SUBSTRINGS:
    sub_mask = df["types"].fillna("").astype(str).apply(
        lambda s: any(sub in s for sub in EXCLUDED_SUBSTRINGS)
    )
    filtered_sub = df.loc[~sub_mask].copy()
    print(f"Substring mode: excluded {sub_mask.sum():,}; {len(filtered_sub):,} left")
    display(filtered_sub.head())
else:
    print("EXCLUDED_SUBSTRINGS is empty — use the token-based `filtered` dataframe above.")

## Export
Uncomment to write a new CSV.

In [ ]:
filtered.to_csv("krakow_leads_filtered.csv", index=False)

## Optional: interactive multi-select (`ipywidgets`)
Pick types to exclude from a list instead of editing a set by hand.

In [10]:
try:
    import ipywidgets as w
except ImportError:
    print("Install with: pip install ipywidgets")
else:
    from IPython.display import display

    _tok = df["types"].dropna().astype(str).str.split("|").explode().str.strip()
    options = sorted(_tok.unique())
    picker = w.SelectMultiple(
        options=options,
        description="Exclude",
        rows=min(18, max(8, len(options) // 4)),
        layout=w.Layout(width="90%", height="320px"),
    )

    out = w.Output()

    def _apply(_=None):
        ex = set(picker.value)
        m = df["types"].apply(lambda v: row_has_excluded_type(v, ex))
        f = df.loc[~m].copy()
        with out:
            out.clear_output()
            print(f"Excluded {m.sum():,} rows; {len(f):,} remaining")
            display(f.head(20))

    btn = w.Button(description="Apply exclusion")
    btn.on_click(_apply)
    display(w.VBox([picker, btn, out]))